# Analyze BBP hippocampus simulation

In this notebook, we will analyze one simulation performed at the Blue Brain Project and published in Romani et al. (2024). This simulation is part of a large set of simulations run with the intent of finding the best parameters to induce theta oscillations in the rat CA1.

The best set of parameters can be summized as follows:
1) External calcium concentration was set to 1 mM to mimic _in vivo_-like conditions
2) All the neurons were stimulationed with a depolarization of 120% (where a stimulation of 100% correspond to the voltage threshold for spike generation)
3) Application of ACh of 1 µM which affects both neuron excitability and synapse release probability
4) PV+ neurons were sitmulated with an oscillatory current of amplitude 0.2 nA and frequency 8 Hz

## Loading

In [ ]:
import obi_auth
from entitysdk.client import Client
from entitysdk.models import Simulation, SimulationCampaign, SimulationResult, SimulationExecution
from entitysdk.common import ProjectContext
from entitysdk import models
from entitysdk.utils.store import LocalAssetStore
from obi_notebook.get_projects import get_projects
from obi_notebook.get_environment import get_environment
from uuid import UUID

from rich import print as rprint
from pathlib import Path
import shutil

import bluepysnap
import matplotlib.pyplot as plt
import numpy as np

Get authentication token and choose project __BBP hippocampus simulations__

In [ ]:
env = "production"

token = obi_auth.get_token(environment=env, auth_mode="daf")

proj = get_projects(token=token, env=env)

Initialize the client with the token and project context

In [ ]:
client = Client(project_context=proj, environment=env, token_manager=token, local_store=LocalAssetStore(prefix="/data"))

In [ ]:
from entitysdk.staging.simulation import stage_simulation
from entitysdk.staging.simulation_result import stage_simulation_result

def stage_single_sim_campaign(simulation_campaign_id, output_dir):
    simulation_campaign = client.get_entity(entity_type=SimulationCampaign, entity_id=simulation_campaign_id)
    simulation_id = simulation_campaign.simulations[0].id
    simulation = client.get_entity(entity_type=Simulation, entity_id=simulation_id)
    
    simulation_execution = client.search_entity(
                entity_type=SimulationExecution,
                query={"used__id": simulation.id}
            ).all()[0]
    
    simulation_result_id = simulation_execution.generated[0].id
    simulation_result = client.get_entity(entity_type=SimulationResult, entity_id=simulation_result_id)
    
    simulation_config_path = stage_simulation_result(
        client=client,
        model=simulation_result,
        output_dir=output_dir,
        simulation_config_file=None,
    )

    return simulation_config_path

In [ ]:
simulation_campaign_id = UUID('47756f30-67c9-4c43-a2f3-75e36318f78f')

simulation_campaign = client.get_entity(entity_type = models.SimulationCampaign, entity_id = simulation_campaign_id)
# rprint(simulation_campaign)

In [ ]:
show_details = True

# Get campaign information
circuit = client.get_entity(entity_id=simulation_campaign.entity_id, entity_type=models.Circuit)
print(f"Loaded campaign '{simulation_campaign.name}' (ID {simulation_campaign.id}):\n")
print(simulation_campaign.description)
print()
print(f"Circuit: '{circuit.name}' (ID {circuit.id})\n")
print(f"Number of simulations: {len(simulation_campaign.simulations)}\n")
print("Scan parameters:")
print("\n".join([f"  {k}: {v}" for k, v in simulation_campaign.scan_parameters.items()]))

# Show campaign details (optional)
if show_details:
    print("\nList of simulations:")
    for sim in simulation_campaign.simulations:
        sim = client.get_entity(entity_id=sim.id, entity_type=models.Simulation)  # To get assets!!
        print(f"  Simulation '{sim.name}' (ID {sim.id}): ")
        print(f"    {sim.scan_parameters}")

In [ ]:
simulation_campaign_id=simulation_campaign.id
output_dir = "./simulation_result_0"

if Path(output_dir).exists():
    shutil.rmtree(Path(output_dir))

simulation_path = stage_single_sim_campaign(simulation_campaign_id, output_dir=output_dir)

In [ ]:
simulation = bluepysnap.Simulation(simulation_path)

In [ ]:
simulation.reports

In [ ]:
soma_report = simulation.reports['soma']

In [ ]:
print(
    soma_report.time_start, 
    soma_report.time_stop, 
    soma_report.dt,
    soma_report.time_units
)

In [ ]:
soma_report.population_names

In [ ]:
soma_pop = soma_report['hippocampus_neurons']

## Raster plot

In [ ]:
spikes = simulation.spikes
spikes.filter().raster()

## Plot traces

In [ ]:
ids = soma_pop.resolve_nodes(group={'synapse_class':'EXC'})


In [ ]:
filtered = soma_report.filter(group={'synapse_class':'EXC'}, t_start=19000, t_stop=20000)
df = filtered.report
df.head()

In [ ]:
ids = df.columns.get_level_values(1).values
n_sample = 10
sel_ids = np.random.choice(ids, n_sample, replace=False)

In [ ]:
df_sampled = df.loc[:, df.columns.get_level_values(1).isin(sel_ids)]
df_sampled.head()

In [ ]:
fig, axs = plt.subplots(n_sample, figsize=(12, 6), sharex=True)

x = df.index.values
for idx, sel_id in enumerate(sel_ids):
    y = df['hippocampus_neurons'][sel_id].values
    axs[idx].plot(x, y)
    if idx == round(n_sample/2):
        axs[idx].set_ylabel('Voltage (mV)')

axs[idx].set_xlabel('Time (ms)')

plt.show()

## References
Romani A, Antonietti A, Bella D, Budd J, Giacalone E, Kurban K, Sáray S, Abdellah M, Arnaudon A, Boci E, Colangelo C, Courcol JD, Delemontex T, Ecker A, Falck J, Favreau C, Gevaert M, Hernando JB, Herttuainen J, Ivaska G, Kanari L, Kaufmann AK, King JG, Kumbhar P, Lange S, Lu H, Lupascu CA, Migliore R, Petitjean F, Planas J, Rai P, Ramaswamy S, Reimann MW, Riquelme JL, Román Guerrero N, Shi Y, Sood V, Sy MF, Van Geit W, Vanherpe L, Freund TF, Mercer A, Muller E, Schürmann F, Thomson AM, Migliore M, Káli S, Markram H. Community-based reconstruction and simulation of a full-scale model of the rat hippocampus CA1 region. PLoS Biol. 2024 Nov 5;22(11):e3002861. doi: 10.1371/journal.pbio.3002861. PMID: 39499732; PMCID: PMC11537418.
https://journals.plos.org/plosbiology/article?id=10.1371/journal.pbio.3002861